# 反欺诈风险评分分析

基于行为日志 CSV 提取 5 个反欺诈特征，加权求和 + 8 条规则增强，输出每用户 0~100 风险评分。

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, '..')
from feature_extraction import extract_features_per_user
from scoring_model import score_risk

# 运行评分
csv_path = "../data/behavior_logs.csv"
per_user = extract_features_per_user(csv_path)

# 构建结果 DataFrame
rows = []
for user_id, features in per_user.items():
    result = score_risk(features)
    dr = features["device_reuse_ratio"]
    ip = features["ip_change_freq"]
    tx = features["tx_freq"]
    lf = features["login_fail_ratio"]
    aa = features["amount_anomaly_score"]

    # 判断哪些规则触发
    rules = []
    if dr > 0.9 and tx > 50:   rules.append("R1")
    if ip > 0.8 and lf > 0.5:  rules.append("R2")
    if aa > 0.7:               rules.append("R3")
    if lf > 0.7:               rules.append("R4")
    if dr > 0.2:               rules.append("R5")
    if ip > 0.1 and aa > 0.3:  rules.append("R6")
    if dr > 0.15 and lf > 0.3: rules.append("R7")
    if tx > 10 and aa > 0.3:   rules.append("R8")

    rows.append({
        "用户": user_id,
        "评分": round(result["score"], 2),
        "等级": result["level"],
        "触发规则": ", ".join(rules) if rules else "-",
        "设备复用": round(dr, 3),
        "IP变更": round(ip, 3),
        "交易频率": round(tx, 1),
        "登录失败率": round(lf, 3),
        "金额异常": round(aa, 3),
    })

df = pd.DataFrame(rows)
df = df.sort_values("评分", ascending=False)
df

In [ ]:
# 带颜色高亮的表格 — 按风险标准标注
def style_table(df):
    # 等级列颜色
    def color_level(val):
        if val == "HIGH":
            return "background-color: #ff4444; color: white; font-weight: bold"
        elif val == "MEDIUM":
            return "background-color: #4da6ff; color: white; font-weight: bold"
        else:
            return "background-color: #28a745; color: white"

    # 触发规则列颜色
    def color_rules(val):
        if val == "-":
            return ""
        n = len(val.split(", "))
        if n >= 4:
            return "background-color: #ff6666; color: white; font-weight: bold"
        elif n >= 2:
            return "background-color: #ffcc66; font-weight: bold"
        else:
            return "background-color: #ffe0b2"

    return df.style\
        .applymap(color_level, subset=["等级"])\
        .applymap(color_rules, subset=["触发规则"])\
        .background_gradient(subset=["设备复用", "IP变更", "登录失败率", "金额异常"],
                           cmap="YlOrRd", vmin=0, vmax=1)\
        .format({"评分": "{:.2f}"})\
        .bar(subset=["评分"], color="#4da6ff", vmin=0, vmax=100)

style_table(df)

## 统计摘要

In [ ]:
print(f"总用户数: {len(df)}")
print(f"低风险 (LOW): {len(df[df['等级']=='LOW'])} 人")
print(f"中风险 (MEDIUM): {len(df[df['等级']=='MEDIUM'])} 人")  
print(f"高风险 (HIGH): {len(df[df['等级']=='HIGH'])} 人")
print(f"平均评分: {df['评分'].mean():.1f}")
print(f"最高评分: {df['评分'].max():.1f} ({df.loc[df['评分'].idxmax(), '用户']})")
print(f"最低评分: {df['评分'].min():.1f} ({df.loc[df['评分'].idxmin(), '用户']})")

## 特征分布

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
features = ["设备复用", "IP变更", "交易频率", "登录失败率", "金额异常"]

for i, feat in enumerate(features):
    low_vals = df[df["等级"] == "LOW"][feat]
    med_vals = df[df["等级"] == "MEDIUM"][feat]
    axes[i].hist([low_vals, med_vals], label=["LOW", "MEDIUM"], bins=10, alpha=0.7)
    axes[i].set_title(feat)
    axes[i].legend()

axes[5].axis("off")
plt.tight_layout()
plt.show()